# EDA - `forecast_balance_data.csv`

Read-only look at the forecasting dataset: shape, dtypes, missing values,
duplicates, and the two things noticed by eye - the new transaction types and
the per-user date ordering.

Nothing here writes anything. Deeper profiling (sentinels, business-key
duplicates, per-column placeholder breakdown) already lives in
`eda/profiler.py`; the last cell points at it.

## Setup

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

SOURCE = Path("data/raw/forecast_balance_data.csv")
print(SOURCE, "-", f"{SOURCE.stat().st_size / 1e6:.1f} MB")

data\raw\forecast_balance_data.csv - 68.3 MB


## Load

Every cell is kept exactly as the file wrote it. `dtype=str` with
`keep_default_na=False` stops pandas turning `""` and `"NA"` into NaN, which
would merge the blank and placeholder categories before we can count them
apart - the same reasoning as `src/utils/io.py`.

One frame, not two. At 265k rows this is ~350 MB; loading a second
type-inferred copy alongside it is enough to exhaust memory on a laptop, and
the next cell gets better answers without it.

In [2]:
raw = pd.read_csv(SOURCE, dtype=str, keep_default_na=False)

print(f"rows: {len(raw):,}   columns: {raw.shape[1]}")
print(f"memory: {raw.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

rows: 265,195   columns: 22
memory: 348 MB


## Columns and datatypes

Rather than asking pandas to guess a dtype per column - which is all-or-nothing,
so one bad cell in 265,000 drops the whole column to text - this measures *what
share* of each column reads as a number and what share reads as a date.

The gap between those two numbers is where the cleaning work is. A column at
95% date and 4% numeric is not a text column; it is a date column with a
different encoding hiding inside it.

In [3]:
import warnings

PROBE = 20_000  # date parsing is the slow part; a sample answers the question


def column_kinds(df: pd.DataFrame, probe: int = PROBE) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        text = df[col].str.strip()
        filled = text[text.ne("")]
        if filled.empty:
            rows.append({
                "column": col, "filled": 0, "numeric_pct": 0.0,
                "date_pct": 0.0, "reads_as": "empty",
            })
            continue

        sample = filled.sample(min(probe, len(filled)), random_state=0)
        numeric = pd.to_numeric(sample, errors="coerce").notna().mean()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            date = pd.to_datetime(
                sample, errors="coerce", format="mixed"
            ).notna().mean()

        # Numeric is tested first on purpose: a 4-digit code like an MCC parses
        # as a year, so a date-first test would label it a date.
        rows.append({
            "column": col,
            "filled": len(filled),
            "numeric_pct": round(100 * numeric, 1),
            "date_pct": round(100 * date, 1),
            "reads_as": (
                "numeric" if numeric > 0.99
                else "date" if date > 0.99
                else "mixed/text"
            ),
        })
    return pd.DataFrame(rows).set_index("column")


kinds = column_kinds(raw)
kinds

,filled,numeric_pct,date_pct,reads_as
column,,,,
USER_ID,265195,0.0,0.0,mixed/text
ACCOUNT_ID,265195,0.0,0.0,mixed/text
TXN_ID,265195,0.0,0.0,mixed/text
TXN_SEQ,265195,100.0,13.5,numeric
TXN_DATE_TIME,265195,4.3,95.7,mixed/text
SETTLE_DATE,264622,0.0,99.8,date
TXN_AMOUNT,265195,99.6,10.4,numeric
TXN_CCY,265195,0.0,0.0,mixed/text
BILLING_AMOUNT,265195,100.0,11.5,numeric


In [4]:
# Columns that are mostly one thing but not entirely -- the cleaning targets.
kinds[
    (kinds["reads_as"] == "mixed/text")
    & (kinds[["numeric_pct", "date_pct"]].max(axis=1) > 1)
]

,filled,numeric_pct,date_pct,reads_as
column,,,,
TXN_DATE_TIME,265195,4.3,95.7,mixed/text
AUTH_CODE,265195,2.2,0.2,mixed/text


Worth pausing on any money or date column that shows up there. `TXN_AMOUNT`
reading as 99.6% numeric means a few hundred rows carry a convention the rest
do not, and `TXN_DATE_TIME` splitting ~96/4 between dates and numbers means a
few thousand timestamps are written as epoch integers rather than dates.

In [5]:
distinct = pd.DataFrame({
    "distinct": [raw[c].nunique() for c in raw.columns],
    "example": [next((v for v in raw[c] if v.strip()), "") for c in raw.columns],
}, index=raw.columns)
distinct.index.name = "column"
distinct

,distinct,example
column,,
USER_ID,150,3cc7e7d7-9b84-566a-b0d0-1553262abc9b
ACCOUNT_ID,355,00b1c46b-3713-51f7-b66d-693b296c2075
TXN_ID,265195,cab394e5-4ced-5831-8767-29ad5d35fdeb
TXN_SEQ,265195,77037
TXN_DATE_TIME,246430,2022-01-01 07:11:25
SETTLE_DATE,4397,03-Jan-22
TXN_AMOUNT,134435,-231.37
TXN_CCY,6,USD
BILLING_AMOUNT,91890,-231.37


## head(10)

In [8]:
raw.head(10)

column,USER_ID,ACCOUNT_ID,TXN_ID,TXN_SEQ,TXN_DATE_TIME,SETTLE_DATE,TXN_AMOUNT,TXN_CCY,BILLING_AMOUNT,BILLING_CURRENCY,FX_RATE,RUNNING_BALANCE,MERCHANT_NAME,MCC_CODE,MERCHANT_COUNTRY,MERCHANT_CITY,PROCESSING_CODE,PROCESSING_TYPE,AUTH_CODE,INTEREST_RATE_INDEX,INFLATION_INDEX,IS_HOLIDAY_MONTH
0,3cc7e7d7-9b84-566a-b0d0-1553262abc9b,00b1c46b-3713-51f7-b66d-693b296c2075,cab394e5-4ced-5831-8767-29ad5d35fdeb,77037,2022-01-01 07:11:25,03-Jan-22,-231.37,USD,-231.37,USD,1.0,,TRM:31659ZAATARWZEIT,5999,LB,BEYRUT,0,PURCHASE,XEYR19,4.519,3.22,False
1,3cc7e7d7-9b84-566a-b0d0-1553262abc9b,00b1c46b-3713-51f7-b66d-693b296c2075,1c019f24-6324-5f9d-9413-e291a094f8df,77038,1640988000,02/01/2022,-99.44,USD,-99.44,USD,1.0,,LULU HYPERMARKET,5411,AE,DUBAYY,0,PURCHASE,VTL9EB,,,
2,3cc7e7d7-9b84-566a-b0d0-1553262abc9b,00b1c46b-3713-51f7-b66d-693b296c2075,244ddbce-9bf0-523f-b522-ba65075b2a7b,77041,2022-01-06 12:58:38,2022-01-07,-207.62,USD,-2.09,USD,1.0044294517,,FACES,5977,AE,DUBAI,0,PURCHASE,WJ2H6X,4.519,0.28,False
3,3cc7e7d7-9b84-566a-b0d0-1553262abc9b,00b1c46b-3713-51f7-b66d-693b296c2075,92ae2fcf-71d6-5fe0-b342-54cf6adeedac,77042,01/06/2022 07:15,2022-01-07,-35.79,USD,-35.95,USD,1.0044294517,,ELECTRICITEDULIBAN-CARDPMT-,4900,LB,BEIRUT LB,0,PURCHASE,A47GJN,4.519,3.22,False
4,3cc7e7d7-9b84-566a-b0d0-1553262abc9b,00b1c46b-3713-51f7-b66d-693b296c2075,2f09a39c-8848-5d70-a9bb-e2e1f940b29f,77043,2022/01/06 03:44,07/01/2022,-162.68,USD,-163.4,USD,1.0044294517,,METROCASHCARRY,5300,FR,PARIS,0,PURCHASE,J366FA,4.519,0.17,False
5,3cc7e7d7-9b84-566a-b0d0-1553262abc9b,00b1c46b-3713-51f7-b66d-693b296c2075,5a469e30-0ba3-59aa-a36f-7c208c88fa74,77044,01/06/2022 22:41,07/01/2022,-151.37,USD,-152.04,USD,1.0044294517,,CARREFOUR UAE,5411,AE,DUBAI,0,PURCHASE,ZUTV1Z,4.519,0.28,False
6,3cc7e7d7-9b84-566a-b0d0-1553262abc9b,00b1c46b-3713-51f7-b66d-693b296c2075,2c6c4a7a-b7ea-57c5-90d0-89790650c08b,77045,01/07/2022 20:40,2022-01-10,-287.75,USD,-289.28,USD,1.0053041065,,lulu hypermarket,5411,AE,DUBAYY,0,PURCHASE,XD283W,4.519,0.28,False
7,3cc7e7d7-9b84-566a-b0d0-1553262abc9b,00b1c46b-3713-51f7-b66d-693b296c2075,9c5e33fa-f4f4-52dc-8a4d-42a4a85ec7c2,77046,2022-01-07 06:13:10,01/07/2022,-84.94,USD,-85.39,USD,1.0053041065,,JUMIA MAROC,5999,MA,CASABLANCA,0,PURCHASE,MFNAQH,4.519,0.54,False
8,3cc7e7d7-9b84-566a-b0d0-1553262abc9b,00b1c46b-3713-51f7-b66d-693b296c2075,0e240986-f5fd-562c-ac6c-1af5e6ccb38e,77047,"Jan 8, 2022",01/10/2022,-124.59,USD,-125.36,USD,1.0061726668,,MEPHICO,5912,LB,,0,PURCHASE,5ZA87F,4.519,3.22,False
9,3cc7e7d7-9b84-566a-b0d0-1553262abc9b,00b1c46b-3713-51f7-b66d-693b296c2075,3098344b-c8e9-5361-a9f9-99c94afac7b4,77048,2022-01-08 03:46:05,01/09/2022,-172.66,USD,-173.73,USD,1.0061726668,,POSMAKEUPSTORE,5977,LB,BEYRUT,0,PURCHASE,HZ0Z8R,4.519,3.22,False


## Missing values

Three separate things get counted, because they are three different problems:

- **blank** - empty cell
- **placeholder** - a string that *means* absent (`NA`, `NULL`, `?`, `0000-00-00`)
- **sentinel** - filler that passes every null check (`00000`, `XXXX`)

The vocabularies are imported from `eda/profiler.py` rather than retyped here,
so the two cannot drift apart.

In [9]:
from eda.profiler import PLACEHOLDERS, SENTINEL_PATTERN


def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        text = df[col].str.strip()
        blank = text.eq("")
        placeholder = ~blank & text.str.upper().isin(PLACEHOLDERS)
        sentinel = ~blank & text.map(lambda v: bool(SENTINEL_PATTERN.match(v)))
        absent = blank | placeholder | sentinel
        rows.append({
            "column": col,
            "blank": int(blank.sum()),
            "placeholder": int(placeholder.sum()),
            "sentinel": int(sentinel.sum()),
            "total_absent": int(absent.sum()),
            "pct": round(100 * absent.mean(), 2),
        })
    return pd.DataFrame(rows).set_index("column").sort_values(
        "total_absent", ascending=False
    )


missing = missing_report(raw)
missing[missing["total_absent"] > 0]

,blank,placeholder,sentinel,total_absent,pct
column,,,,,
PROCESSING_CODE,0,0,203248,203248,76.64
RUNNING_BALANCE,92857,0,0,92857,35.01
MERCHANT_CITY,46550,0,0,46550,17.55
INFLATION_INDEX,29150,0,0,29150,10.99
IS_HOLIDAY_MONTH,11277,0,0,11277,4.25
INTEREST_RATE_INDEX,11277,0,0,11277,4.25
AUTH_CODE,0,0,5691,5691,2.15
SETTLE_DATE,573,543,0,1116,0.42
TXN_SEQ,0,0,5,5,0.00


In [10]:
clean_columns = missing[missing["total_absent"] == 0].index.tolist()
print(f"{len(clean_columns)} columns with nothing absent:")
print(clean_columns)

13 columns with nothing absent:
['ACCOUNT_ID', 'TXN_DATE_TIME', 'TXN_ID', 'USER_ID', 'TXN_AMOUNT', 'MCC_CODE', 'MERCHANT_NAME', 'FX_RATE', 'BILLING_CURRENCY', 'TXN_CCY', 'BILLING_AMOUNT', 'PROCESSING_TYPE', 'MERCHANT_COUNTRY']


### Do the gaps line up?

Missing values that co-occur are one defect, not several. If a set of columns is
always blank on the same rows, that is a single upstream cause and it gets
handled once - so the pattern matters more than the per-column totals above.

In [11]:
gappy = missing[missing["total_absent"] > 0].index.tolist()
blank_map = pd.DataFrame(
    {col: raw[col].str.strip().eq("") for col in gappy}
)

patterns = (
    blank_map.groupby(gappy, sort=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)
print(f"{len(patterns)} distinct missingness patterns across {len(gappy)} columns")
patterns.head(15)

24 distinct missingness patterns across 9 columns


,PROCESSING_CODE,RUNNING_BALANCE,MERCHANT_CITY,INFLATION_INDEX,IS_HOLIDAY_MONTH,INTEREST_RATE_INDEX,AUTH_CODE,SETTLE_DATE,TXN_SEQ,rows
6,False,False,False,False,False,False,False,False,False,124789
0,False,True,False,False,False,False,False,False,False,67727
8,False,False,True,False,False,False,False,False,False,28265
2,False,True,True,False,False,False,False,False,False,14753
9,False,False,False,True,False,False,False,False,False,10623
7,False,False,False,True,True,True,False,False,False,6061
3,False,True,False,True,False,False,False,False,False,5698
1,False,True,False,True,True,True,False,False,False,3278
10,False,False,True,True,True,True,False,False,False,1249
12,False,False,True,True,False,False,False,False,False,995


## Duplicates

Three questions, not one: is the whole row repeated, is the identifier
repeated, and is the same *transaction* present under two different ids? The
business keys match `config/policy.yaml`, so what counts as a duplicate here is
what will count as one in the pipeline.

In [12]:
print(f"exact duplicate rows : {raw.duplicated().sum():,}")

for col in ["TXN_ID", "TXN_SEQ"]:
    if col in raw.columns:
        print(f"repeated {col:<8}    : {raw[col].duplicated().sum():,}")

BUSINESS_KEYS = [
    ["ACCOUNT_ID", "TXN_DATE_TIME", "TXN_AMOUNT", "MERCHANT_NAME"],
    ["ACCOUNT_ID", "TXN_DATE_TIME", "TXN_AMOUNT"],
]
for key in BUSINESS_KEYS:
    if all(c in raw.columns for c in key):
        n = raw.duplicated(subset=key).sum()
        print(f"repeats on {'+'.join(key)}: {n:,}")

exact duplicate rows : 0
repeated TXN_ID      : 0
repeated TXN_SEQ     : 0
repeats on ACCOUNT_ID+TXN_DATE_TIME+TXN_AMOUNT+MERCHANT_NAME: 0
repeats on ACCOUNT_ID+TXN_DATE_TIME+TXN_AMOUNT: 0


## Cardinality

Which columns are identifiers, which are categories, and which are free text.
Anything with a handful of distinct values is a vocabulary the pipeline needs a
rule file for.

In [13]:
card = distinct["distinct"].sort_values()
card.to_frame()

,distinct
column,
BILLING_CURRENCY,1
IS_HOLIDAY_MONTH,3
TXN_CCY,6
MERCHANT_COUNTRY,12
PROCESSING_CODE,13
PROCESSING_TYPE,13
MCC_CODE,26
MERCHANT_CITY,40
INTEREST_RATE_INDEX,42


In [14]:
LOW_CARDINALITY = 30
for col in card[card <= LOW_CARDINALITY].index:
    print(f"--- {col} ---")
    print(raw[col].value_counts(dropna=False).to_string())
    print()

--- BILLING_CURRENCY ---
BILLING_CURRENCY
USD    265195

--- IS_HOLIDAY_MONTH ---
IS_HOLIDAY_MONTH
False    216222
True      37696
          11277

--- TXN_CCY ---
TXN_CCY
USD    194151
LBP     36647
AED     16474
MAD      9894
ARS      6659
EUR      1370

--- MERCHANT_COUNTRY ---
MERCHANT_COUNTRY
LB    172435
AE     33466
US     21612
MA      9525
DE      4774
FR      4709
SE      4595
GB      4499
EE      2500
TR      2404
NL      2394
ES      2282

--- PROCESSING_CODE ---
PROCESSING_CODE
0     203248
26     27417
1       7168
23      6929
21      5896
22      4959
25      4207
27      1977
24      1586
30       604
31       594
29       309
28       301

--- PROCESSING_TYPE ---
PROCESSING_TYPE
PURCHASE             203248
SETTLEMENT_CREDIT     27417
ATM_WITHDRAWAL         7168
TRANSFER_OUT           6929
SALARY_CREDIT          5896
TRANSFER_IN            4959
INTEREST               4207
CARD_PAYMENT           1977
FEE                    1586
INSURANCE_PREMIUM       604
EMERGENCY_EXPE

## Transaction types

The observation about new types, quantified. `PROCESSING_CODE` and
`PROCESSING_TYPE` are two spellings of one fact, so the useful view is the
pairing: it shows whether they agree 1:1, and which codes are new relative to
`src/rules/json/processing_codes.json`.

In [15]:
pair = (
    raw.groupby(["PROCESSING_CODE", "PROCESSING_TYPE"])
    .size()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)
print(f"distinct code->type pairs: {len(pair)}")
print(f"distinct codes           : {raw['PROCESSING_CODE'].nunique()}")
pair

distinct code->type pairs: 13
distinct codes           : 13


,PROCESSING_CODE,PROCESSING_TYPE,rows
0,0,PURCHASE,203248
7,26,SETTLEMENT_CREDIT,27417
1,1,ATM_WITHDRAWAL,7168
4,23,TRANSFER_OUT,6929
2,21,SALARY_CREDIT,5896
3,22,TRANSFER_IN,4959
6,25,INTEREST,4207
8,27,CARD_PAYMENT,1977
5,24,FEE,1586
11,30,INSURANCE_PREMIUM,604


In [16]:
import json

known = json.loads(
    Path("src/rules/json/processing_codes.json").read_text(encoding="utf-8")
)["codes"]
WIDTH = 2  # codes.processing_code_width in config/policy.yaml

present = {str(c).strip().zfill(WIDTH) for c in raw["PROCESSING_CODE"].unique()}
new = sorted(present - set(known))
unused = sorted(set(known) - present)

print("known in rule file :", sorted(known))
print("new in this file   :", new)
print("in rules, not here :", unused)

known in rule file : ['00', '01', '20']
new in this file   : ['21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31']
in rules, not here : ['20']


In [17]:
# The label each new code carries in this file - what the rule file would need
# to say. Shown, not written: extending the vocabulary is a deliberate edit,
# not something a notebook should do behind your back.
labels = (
    raw.assign(CODE=raw["PROCESSING_CODE"].str.strip().str.zfill(WIDTH))
    .loc[lambda d: d["CODE"].isin(new)]
    .groupby("CODE")["PROCESSING_TYPE"]
    .agg(["unique", "size"])
    .rename(columns={"unique": "labels_seen", "size": "rows"})
)
labels

,labels_seen,rows
CODE,,
21,[SALARY_CREDIT],5896
22,[TRANSFER_IN],4959
23,[TRANSFER_OUT],6929
24,[FEE],1586
25,[INTEREST],4207
26,[SETTLEMENT_CREDIT],27417
27,[CARD_PAYMENT],1977
28,[BONUS_CREDIT],301
29,[TUITION],309


## Date ordering

Testing the observation directly: is the file grouped by user, and is each
user's block in date order?

The parse here is **lenient on purpose** - it is for looking at ordering, not
for cleaning. The pipeline's strict, format-by-format parser is the one whose
failures get counted; this one only needs to sort.

In [18]:
def lenient_datetime(series: pd.Series) -> pd.Series:
    '''Parses for exploration only: epoch seconds, then anything pandas reads.'''
    text = series.astype(str).str.strip()
    epoch = text.str.fullmatch(r"\d{9,10}")
    out = pd.to_datetime(text.where(~epoch), errors="coerce", format="mixed")
    out.loc[epoch] = pd.to_datetime(
        pd.to_numeric(text[epoch], errors="coerce"), unit="s"
    )
    return out


when = lenient_datetime(raw["TXN_DATE_TIME"])
print(f"unparsed even leniently: {when.isna().sum():,}")
print(f"range: {when.min()}  ->  {when.max()}")

unparsed even leniently: 0
range: 2021-12-31 22:00:00  ->  2025-06-30 23:55:00


If the range starts a couple of hours before midnight on New Year rather than
on it, that is the epoch rows: `unit="s"` reads them as UTC while the written
timestamps carry local time. Worth knowing before either one is trusted as an
index - but a cleaning decision, not an EDA one.

In [19]:
# Is each user's block contiguous, or are their rows scattered through the file?
users = raw["USER_ID"].nunique()
blocks = raw["USER_ID"].ne(raw["USER_ID"].shift()).cumsum().nunique()

print(f"distinct users        : {users:,}")
print(f"contiguous user blocks: {blocks:,}")
print(
    "-> each user's rows sit together"
    if blocks == users
    else "-> user rows are interleaved"
)

distinct users        : 150
contiguous user blocks: 663
-> user rows are interleaved


In [20]:
# Within each block, does time only move forward? Checked per user and per
# account, since one user holds several accounts.
frame = raw.assign(_when=when, _pos=range(len(raw)))


def monotonic_share(group_col: str) -> None:
    ok = (
        frame.dropna(subset=["_when"])
        .sort_values("_pos")
        .groupby(group_col)["_when"]
        .apply(lambda s: s.is_monotonic_increasing)
    )
    print(
        f"{group_col:<11}: {ok.sum():,} of {len(ok):,} in date order "
        f"({100 * ok.mean():.1f}%)"
    )


monotonic_share("USER_ID")
monotonic_share("ACCOUNT_ID")

USER_ID    : 2 of 150 in date order (1.3%)
ACCOUNT_ID : 64 of 355 in date order (18.0%)


In [21]:
# TXN_SEQ is the other candidate ordering. Where it is monotonic and the dates
# are not, it is the more trustworthy sort key - which matters for the
# ambiguous slash-format dates, where sequence can break the tie.
seq = pd.to_numeric(raw["TXN_SEQ"], errors="coerce")
frame = frame.assign(_seq=seq)

by_seq = (
    frame.dropna(subset=["_seq"])
    .sort_values("_pos")
    .groupby("ACCOUNT_ID")["_seq"]
    .apply(lambda s: s.is_monotonic_increasing)
)
print(f"TXN_SEQ monotonic within account : {by_seq.sum():,} of {len(by_seq):,}")
print(f"TXN_SEQ monotonic in file order  : {seq.is_monotonic_increasing}")

TXN_SEQ monotonic within account : 355 of 355
TXN_SEQ monotonic in file order  : False


### Reading those two results together

If `TXN_SEQ` is monotonic within every account but the parsed dates are not,
that is **evidence about the parse, not about the data**. The lenient parser
above guesses on `01/06/2022`-style strings, and this file contains both
day-first and month-first spellings of that shape - so a row landing out of
order is most likely a date read with its day and month swapped.

Which makes `TXN_SEQ` the more trustworthy ordering, and gives the cleaning
stage a way to settle the ambiguous dates: of the two readings, prefer the one
that keeps the account's timeline moving forward.

The cell below counts how big that effect is - rows whose neighbours in
sequence order sit on either side of them in time.

In [22]:
ordered = frame.dropna(subset=["_seq", "_when"]).sort_values(["ACCOUNT_ID", "_seq"])
backwards = (
    ordered.groupby("ACCOUNT_ID")["_when"].diff().dt.total_seconds().lt(0)
)
print(f"rows that move time backwards in sequence order: {backwards.sum():,} "
      f"({100 * backwards.mean():.1f}%)")

# Are they concentrated in the ambiguous slash format, as the theory predicts?
shape = (
    raw.loc[backwards[backwards].index, "TXN_DATE_TIME"]
    .str.replace(r"\d", "9", regex=True)
    .str.replace(r"[A-Za-z]", "A", regex=True)
    .value_counts()
)
shape.head(10).to_frame("rows_out_of_order")

rows that move time backwards in sequence order: 44,698 (16.9%)


,rows_out_of_order
TXN_DATE_TIME,
9999-99-99 99:99:99,15367
99/99/9999 99:99,14817
9999999999,3651
99-AAA-99 99:99,3440
9999/99/99 99:99,3328
"AAA 99, 9999",2919
"AAA 9, 9999",1176


In [23]:
# The first user's block, as the observation described it.
first_user = raw["USER_ID"].iloc[0]
sample = frame[frame["USER_ID"] == first_user].sort_values("_pos")
print(f"user {first_user} - {len(sample):,} rows")
sample[["_when", "TXN_SEQ", "TXN_AMOUNT", "PROCESSING_TYPE", "MERCHANT_NAME"]].head(15)

user 3cc7e7d7-9b84-566a-b0d0-1553262abc9b - 1,752 rows


column,_when,TXN_SEQ,TXN_AMOUNT,PROCESSING_TYPE,MERCHANT_NAME
0,2022-01-01 07:11:25,77037,-231.37,PURCHASE,TRM:31659ZAATARWZEIT
1,2021-12-31 22:00:00,77038,-99.44,PURCHASE,LULU HYPERMARKET
2,2022-01-06 12:58:38,77041,-207.62,PURCHASE,FACES
3,2022-01-06 07:15:00,77042,-35.79,PURCHASE,ELECTRICITEDULIBAN-CARDPMT-
4,2022-01-06 03:44:00,77043,-162.68,PURCHASE,METROCASHCARRY
5,2022-01-06 22:41:00,77044,-151.37,PURCHASE,CARREFOUR UAE
6,2022-01-07 20:40:00,77045,-287.75,PURCHASE,lulu hypermarket
7,2022-01-07 06:13:10,77046,-84.94,PURCHASE,JUMIA MAROC
8,2022-01-08 00:00:00,77047,-124.59,PURCHASE,MEPHICO
9,2022-01-08 03:46:05,77048,-172.66,PURCHASE,POSMAKEUPSTORE


In [24]:
# Month-by-month volume for that user, which is what "January then February"
# should look like if the block really is chronological.
dated = sample.dropna(subset=["_when"])
dated.groupby(dated["_when"].dt.to_period("M")).size().rename("rows").to_frame()

,rows
_when,
2021-12,1
2022-01,56
2022-02,53
2022-03,32
2022-04,47
2022-05,40
2022-06,48
2022-07,45
2022-08,34


## Numeric spread

`TXN_AMOUNT` arrives as text with mixed conventions, so this is a lenient read
purely to see the shape and spot obvious outliers.

In [25]:
amount = pd.to_numeric(raw["TXN_AMOUNT"], errors="coerce")
print(f"unparsed as plain numeric: {amount.isna().sum():,}")
amount.describe()

unparsed as plain numeric: 1,126


count    2.640690e+05
mean     6.607942e+04
std      1.117726e+08
min     -1.165997e+10
25%     -3.634900e+02
50%     -1.204000e+02
75%     -4.013000e+01
max      2.938623e+09
Name: TXN_AMOUNT, dtype: float64

In [26]:
raw.assign(_amt=amount).groupby("PROCESSING_TYPE")["_amt"].agg(
    ["size", "min", "median", "max"]
).sort_values("size", ascending=False)

,size,min,median,max
PROCESSING_TYPE,,,,
PURCHASE,203248,-1.165997e+10,-145.270,-0.000000e+00
SETTLEMENT_CREDIT,27417,4.950000e+00,2877.420,2.938623e+09
ATM_WITHDRAWAL,7168,-4.862368e+07,-359.795,-2.280000e+00
TRANSFER_OUT,6929,-3.526151e+08,-1801.160,-3.300000e+00
SALARY_CREDIT,5896,2.246000e+01,5482.210,4.228672e+08
TRANSFER_IN,4959,2.990000e+00,1101.620,3.074990e+08
INTEREST,4207,0.000000e+00,66.050,7.101210e+05
CARD_PAYMENT,1977,1.284000e+01,2439.240,3.526151e+08
FEE,1586,-8.562685e+07,-48.120,-8.900000e-01


## Next

`eda/profiler.py` goes further than this - sentinel detection per column,
duplicate detection across several business keys, and a sampled view of the
offending rows:

```python
from eda.profiler import DataProfiler
DataProfiler(SOURCE).load().report()
```